# Relatório – Filtragem Passa-Baixa em Imagens Digitais

**Autores:**
- Igor Ladeia de Freitas (RA11201922180)
- Gustavo Fernandes do Nascimento (RA11202021700)
- Ryan Lucas da Silva (RA11202522362)
- Eduardo Yukio Makita (RA11202020221)

**Data de realização:** 13/03/2026  
**Data de publicação:** 17/03/2026

---

---

## 1. Introdução

A filtragem de imagens é uma técnica fundamental no processamento digital de imagens, sendo amplamente utilizada para redução de ruídos e suavização de detalhes indesejados.

Os filtros passa-baixa têm como principal objetivo suavizar a imagem, reduzindo variações bruscas de intensidade entre pixels vizinhos. Entre os principais métodos estudados neste trabalho estão:

- Filtro de média (blur)
- Filtro Gaussiano
- Filtro de mediana
- Filtro bilateral

Cada um desses filtros possui características próprias, sendo mais ou menos eficaz dependendo do tipo de ruído presente na imagem.

Neste relatório, serão realizados experimentos aplicando esses filtros em imagens com e sem ruído, além da avaliação quantitativa da qualidade das imagens filtradas.

## 2. Procedimentos Experimentais

### 2.1 Aplicação dos filtros passa-baixa

Nesta etapa, foi utilizada uma imagem base do grupo para aplicação de diferentes filtros passa-baixa.

Foram aplicados os seguintes filtros:
- Média (blur)
- Gaussiano (GaussianBlur)
- Mediana (medianBlur)
- Bilateral (bilateralFilter)

Além disso, foram testados diferentes tamanhos de kernel: 3x3, 5x5, 11x11 e 29x29.

O objetivo é analisar como o tamanho do kernel influencia no nível de suavização da imagem. Kernels maiores tendem a gerar maior borramento, porém podem remover mais ruído.

As imagens resultantes são salvas em formato `.jpg` para posterior análise.

In [ ]:
import cv2
import numpy as np
import os

img = cv2.imread('imagem_grupo.png')

kernels = [3, 5, 11, 29]
filtros = ['Blur', 'Gaussian', 'Median', 'Bilateral']

for k in kernels:

    res_blur = cv2.blur(img, (k, k))
    cv2.imwrite(f'resultado_blur_k{k}.jpg', res_blur)

    res_gauss = cv2.GaussianBlur(img, (k, k), 0)
    cv2.imwrite(f'resultado_gauss_k{k}.jpg', res_gauss)

    res_median = cv2.medianBlur(img, k)
    cv2.imwrite(f'resultado_median_k{k}.jpg', res_median)

    res_bilat = cv2.bilateralFilter(img, k, 75, 75)
    cv2.imwrite(f'resultado_bilat_k{k}.jpg', res_bilat)

### 2.2 Inserção de ruído nas imagens

Nesta etapa, foram geradas imagens com ruído artificial para simular cenários reais de degradação.

Foram utilizados dois tipos de ruído:

- **Ruído Gaussiano:** caracterizado por variações aleatórias seguindo uma distribuição normal.
- **Ruído Sal e Pimenta:** caracterizado por pixels aleatórios que assumem valores extremos (preto ou branco).

Esses ruídos foram adicionados com intensidade elevada, criando imagens de teste desafiadoras para os filtros.

O objetivo é avaliar a capacidade dos filtros em recuperar a qualidade da imagem original.

In [ ]:
def add_gaussian_noise(image):
    row, col, ch = image.shape
    mean = 0
    var = 500
    sigma = var**0.5
    gauss = np.random.normal(mean, sigma, (row, col, ch))
    noisy = image + gauss
    return np.clip(noisy, 0, 255).astype(np.uint8)

def add_salt_pepper_noise(image, amount=0.1): # 10% da imagem com ruído
    out = np.copy(image)
    num_salt = np.ceil(amount * image.size * 0.5)
    coords = [np.random.randint(0, i - 1, int(num_salt)) for i in image.shape]
    out[tuple(coords)] = 255
    num_pepper = np.ceil(amount * image.size * 0.5)
    coords = [np.random.randint(0, i - 1, int(num_pepper)) for i in image.shape]
    out[tuple(coords)] = 0
    return out

img_gauss_ruido = add_gaussian_noise(img)
img_sp_ruido = add_salt_pepper_noise(img)

### 2.3 Aplicação dos filtros nas imagens com ruído

Após a geração das imagens com ruído, foram aplicados novamente os filtros passa-baixa utilizando os mesmos tamanhos de kernel.

Essa etapa permite comparar o desempenho dos filtros em cenários degradados e verificar qual técnica é mais eficiente para cada tipo de ruído.

In [ ]:
import pandas as pd
from skimage.metrics import structural_similarity as ssim

dados_metricas = []

ruidos = {
    'Gaussiano': img_gauss_ruido,
    'Sal e Pimenta': img_sp_ruido
}

print("Processando filtragens e métricas... aguarde.")

for nome_ruido, img_ruidosa in ruidos.items():
    for k in kernels:
        filtros_processados = {
            'Media': cv2.blur(img_ruidosa, (k, k)),
            'Gaussiano': cv2.GaussianBlur(img_ruidosa, (k, k), 0),
            'Mediana': cv2.medianBlur(img_ruidosa, k),
            'Bilateral': cv2.bilateralFilter(img_ruidosa, k, 75, 75)
        }

        for nome_filtro, img_filtrada in filtros_processados.items():
            psnr_v = cv2.PSNR(img, img_filtrada)
            ssim_v = ssim(img, img_filtrada, channel_axis=2)

            nome_arquivo = f"filt_{nome_ruido}_{nome_filtro}_k{k}.jpg"
            cv2.imwrite(nome_arquivo, img_filtrada)

            dados_metricas.append([nome_ruido, nome_filtro, k, psnr_v, ssim_v])

df_relatorio = pd.DataFrame(dados_metricas, columns=['Tipo Ruído', 'Filtro', 'Kernel', 'PSNR', 'SSIM'])
pd.set_option('display.max_rows', None) # Para ver todas as linhas
display(df_relatorio)

Processando filtragens e métricas... aguarde.


,Tipo Ruído,Filtro,Kernel,PSNR,SSIM
0,Gaussiano,Media,3,27.747893,0.657841
1,Gaussiano,Gaussiano,3,28.090841,0.618361
2,Gaussiano,Mediana,3,28.194599,0.576429
3,Gaussiano,Bilateral,3,26.686043,0.477537
4,Gaussiano,Media,5,26.650228,0.790914
5,Gaussiano,Gaussiano,5,28.332736,0.732815
6,Gaussiano,Mediana,5,29.825010,0.751686
7,Gaussiano,Bilateral,5,29.542541,0.639949
8,Gaussiano,Media,11,23.959630,0.783612
9,Gaussiano,Gaussiano,11,26.502242,0.841439


### 2.4 Avaliação de qualidade das imagens

Para avaliar a eficácia dos filtros, foram utilizadas métricas objetivas de qualidade de imagem:

- **PSNR (Peak Signal-to-Noise Ratio):**
  Mede a relação entre o sinal original e o ruído. Quanto maior o valor, melhor a qualidade da imagem.

- **SSIM (Structural Similarity Index):**
  Avalia a similaridade estrutural entre duas imagens. Valores próximos de 1 indicam alta similaridade.

Essas métricas permitem comparar quantitativamente o desempenho dos diferentes filtros aplicados nas imagens com ruído.

In [ ]:
from skimage.metrics import structural_similarity as ssim

resultados_metricas = []

for k in kernels:
    filtrada = cv2.medianBlur(img_sp_ruido, k)

    psnr_val = cv2.PSNR(img, filtrada)

    ssim_val = ssim(img, filtrada, channel_axis=2)

    resultados_metricas.append((k, psnr_val, ssim_val))
    print(f"Kernel {k}x{k} -> PSNR: {psnr_val:.2f}, SSIM: {ssim_val:.4f}")

Kernel 3x3 -> PSNR: 34.65, SSIM: 0.9695
Kernel 5x5 -> PSNR: 33.35, SSIM: 0.9521
Kernel 11x11 -> PSNR: 27.57, SSIM: 0.8820
Kernel 29x29 -> PSNR: 22.62, SSIM: 0.7934


### 2.6 Aplicação em tempo real com webcam

Nesta etapa, foi desenvolvido um programa que captura imagens em tempo real da webcam e aplica filtros de suavização continuamente.

O sistema permite:

- Visualizar a imagem filtrada em tempo real
- Salvar imagens pressionando a tecla **[s]**
- Encerrar a execução com a tecla **[q]**

Foram escolhidos dois tipos de filtros com base nos melhores e piores resultados observados anteriormente.

Esse experimento demonstra o uso prático dos filtros em aplicações em tempo real.

In [ ]:
import cv2

# Inicializa a webcam
cap = cv2.VideoCapture(0)

print("Controles: [s] Salvar imagens | [q] Sair")

while True:
    ret, frame = cap.read()
    if not ret:
        break

       melhor_img = cv2.medianBlur(frame, 3)
    pior_img = cv2.blur(frame, (29, 29))

    cv2.imshow('Webcam Original', frame)
    cv2.imshow('Melhor Resultado (Mediana 3x3)', melhor_img)
    cv2.imshow('Pior Resultado (Blur 29x29)', pior_img)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('s'):
        cv2.imwrite('webcam_melhor.jpg', melhor_img)
        cv2.imwrite('webcam_pior.jpg', pior_img)
        print("Imagens da webcam salvas!")

    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Controles: [s] Salvar imagens | [q] Sair


## 3. Análise e Discussão

Os resultados mostraram que diferentes filtros possuem desempenhos distintos dependendo do tipo de ruído:

- O **filtro de mediana** apresentou melhor desempenho para ruído sal e pimenta, preservando melhor os contornos.
- O **filtro Gaussiano** teve bom desempenho para ruído gaussiano, promovendo uma suavização mais natural.
- O **filtro de média** apresentou maior borramento, sendo menos eficiente na preservação de detalhes.
- O **filtro bilateral** conseguiu preservar bordas, porém com maior custo computacional.

Além disso, o aumento do tamanho do kernel resultou em maior suavização, porém com perda de detalhes importantes da imagem.

A análise das métricas PSNR e SSIM confirmou essas observações.

## 4. Conclusão

Os filtros passa-baixa são ferramentas importantes no processamento de imagens, especialmente para remoção de ruídos.

Cada filtro possui vantagens específicas:

- Mediana: ideal para ruído impulsivo (sal e pimenta)
- Gaussiano: adequado para ruído distribuído
- Bilateral: preserva bordas, porém é mais custoso
- Média: simples, porém menos eficiente

A escolha do filtro ideal depende do tipo de ruído e do objetivo da aplicação.

Este trabalho permitiu compreender na prática o funcionamento e as limitações desses filtros.

## 5. Referências

- OpenCV Documentation – Smoothing Images  
https://docs.opencv.org/4.x/dc/dd3/tutorial_gausian_median_blur_bilateral_filter.html

- OpenCV – PSNR e SSIM  
https://docs.opencv.org/4.x/d5/dc4/tutorial_video_input_psnr_ssim.html

- AskPython – Adding noise to images  
https://www.askpython.com/python/examples/adding-noise-imagee486a0aa1f60